In [ ]:
"""
Task 1: Python Fundamentals
----------------------------
Five problems demonstrating functions, loops, and list operations.
Each problem is implemented as a standalone, testable function.
Run this file directly to see sample output for every problem.
"""


# ---------------------------------------------------------------------------
# Problem 1: Prime numbers in a range
# ---------------------------------------------------------------------------
def find_primes(start: int, end: int) -> list[int]:
    """Return all prime numbers in the inclusive range [start, end]."""
    primes = []
    for num in range(max(start, 2), end + 1):
        is_prime = True
        for divisor in range(2, int(num ** 0.5) + 1):
            if num % divisor == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(num)
    return primes


# ---------------------------------------------------------------------------
# Problem 2: Word frequency counter
# ---------------------------------------------------------------------------
def word_frequency(text: str) -> dict[str, int]:
    """Return a dictionary mapping each word (lowercased) to its frequency."""
    freq = {}
    words = text.lower().split()
    for word in words:
        cleaned = "".join(ch for ch in word if ch.isalnum())
        if not cleaned:
            continue
        freq[cleaned] = freq.get(cleaned, 0) + 1
    return freq


# ---------------------------------------------------------------------------
# Problem 3: List statistics (without using built-in statistics module)
# ---------------------------------------------------------------------------
def list_statistics(numbers: list[float]) -> dict[str, float]:
    """Return min, max, mean, and median of a list of numbers."""
    if not numbers:
        raise ValueError("Cannot compute statistics on an empty list")

    sorted_nums = sorted(numbers)
    n = len(sorted_nums)
    mean_val = sum(sorted_nums) / n

    if n % 2 == 0:
        median_val = (sorted_nums[n // 2 - 1] + sorted_nums[n // 2]) / 2
    else:
        median_val = sorted_nums[n // 2]

    return {
        "min": sorted_nums[0],
        "max": sorted_nums[-1],
        "mean": round(mean_val, 2),
        "median": median_val,
    }


# ---------------------------------------------------------------------------
# Problem 4: Remove duplicates while preserving order
# ---------------------------------------------------------------------------
def remove_duplicates(items: list) -> list:
    """Return a new list with duplicates removed, preserving first occurrence order."""
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


# ---------------------------------------------------------------------------
# Problem 5: Fibonacci sequence generator (using a loop, not recursion)
# ---------------------------------------------------------------------------
def fibonacci(n_terms: int) -> list[int]:
    """Return the first n_terms of the Fibonacci sequence."""
    if n_terms <= 0:
        return []
    sequence = [0, 1][:n_terms]
    for i in range(2, n_terms):
        sequence.append(sequence[-1] + sequence[-2])
    return sequence


# ---------------------------------------------------------------------------
# Demonstration
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    print("Problem 1: Primes between 10 and 50")
    print(find_primes(10, 50))
    print()

    print("Problem 2: Word frequency")
    sample_text = "the quick brown fox jumps over the lazy dog the fox runs"
    print(word_frequency(sample_text))
    print()

    print("Problem 3: List statistics")
    sample_numbers = [12, 45, 7, 23, 56, 89, 34, 2, 67]
    print(list_statistics(sample_numbers))
    print()

    print("Problem 4: Remove duplicates")
    sample_list = [1, 2, 2, 3, 4, 4, 4, 5, 1, 6]
    print(remove_duplicates(sample_list))
    print()

    print("Problem 5: First 12 Fibonacci numbers")
    print(fibonacci(12))

Problem 1: Primes between 10 and 50
[11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]

Problem 2: Word frequency
{'the': 3, 'quick': 1, 'brown': 1, 'fox': 2, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1, 'runs': 1}

Problem 3: List statistics
{'min': 2, 'max': 89, 'mean': 37.22, 'median': 34}

Problem 4: Remove duplicates
[1, 2, 3, 4, 5, 6]

Problem 5: First 12 Fibonacci numbers
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Adjust paths for Colab environment
# Assumes 'student_performance.csv' is directly in the current working directory (/content/)
DATA_PATH = Path.cwd() / "student_performance.csv"
OUTPUT_DIR = Path.cwd()


def load_data(path: Path) -> pd.DataFrame:
    """Load the CSV into a DataFrame."""
    df = pd.read_csv(path)
    return df


def inspect_raw(df: pd.DataFrame) -> None:
    """Print shape, dtypes, and null counts before cleaning."""
    print("=" * 60)
    print("RAW DATA INSPECTION")
    print("=" * 60)
    print(f"Shape (rows, columns): {df.shape}")
    print("\nColumn dtypes:")
    print(df.dtypes)
    print("\nMissing values per column:")
    print(df.isnull().sum())
    print(f"\nDuplicate rows: {df.duplicated().sum()}")


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the dataset:
      - Drop exact duplicate rows
      - Impute missing numeric values with the column median
        (median is more robust to outliers than the mean)
    """
    df_clean = df.drop_duplicates().copy()

    numeric_cols = df_clean.select_dtypes(include="number").columns
    for col in numeric_cols:
        if df_clean[col].isnull().any():
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)

    return df_clean.reset_index(drop=True)


def summarize(df: pd.DataFrame) -> None:
    """Print basic statistics: shape, nulls, and mean for numeric columns."""
    print("\n" + "=" * 60)
    print("CLEANED DATA SUMMARY")
    print("=" * 60)
    print(f"Shape (rows, columns): {df.shape}")
    print("\nMissing values per column (should be 0 everywhere):")
    print(df.isnull().sum())

    print("\nMean of each numeric column:")
    print(df.select_dtypes(include="number").mean().round(2))

    print("\nFull descriptive statistics:")
    print(df.describe().round(2))

    print("\nExtracurricular participation counts:")
    print(df["extracurricular"].value_counts())


def make_visualizations(df: pd.DataFrame) -> None:
    """Save a correlation heatmap and a distribution plot for the README."""
    sns.set_style("whitegrid")

    # Correlation heatmap
    plt.figure(figsize=(7, 5))
    numeric_df = df.select_dtypes(include="number").drop(columns=["student_id"], errors="ignore")
    sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Heatmap - Student Performance")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=150)
    plt.close()

    # Final score distribution
    plt.figure(figsize=(7, 5))
    sns.histplot(df["final_score"], kde=True, bins=20, color="steelblue")
    plt.title("Distribution of Final Scores")
    plt.xlabel("Final Score")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "final_score_distribution.png", dpi=150)
    plt.close()

    print(f"\nSaved plots to {OUTPUT_DIR}")


def main():
    df_raw = load_data(DATA_PATH)
    inspect_raw(df_raw)

    df_clean = clean_data(df_raw)
    summarize(df_clean)

    make_visualizations(df_clean)

    # Save cleaned dataset for downstream use (e.g., Task 3)
    df_clean.to_csv(OUTPUT_DIR / "student_performance_clean.csv", index=False)
    print(f"\nCleaned dataset saved to {OUTPUT_DIR / 'student_performance_clean.csv'}")


if __name__ == "__main__":
    main()

RAW DATA INSPECTION
Shape (rows, columns): (305, 7)

Column dtypes:
student_id              object
study_hours_per_day    float64
attendance_percent     float64
sleep_hours            float64
previous_score         float64
extracurricular         object
final_score            float64
dtype: object

Missing values per column:
student_id             0
study_hours_per_day    0
attendance_percent     9
sleep_hours            4
previous_score         2
extracurricular        0
final_score            0
dtype: int64

Duplicate rows: 5

CLEANED DATA SUMMARY
Shape (rows, columns): (300, 7)

Missing values per column (should be 0 everywhere):
student_id             0
study_hours_per_day    0
attendance_percent     0
sleep_hours            0
previous_score         0
extracurricular        0
final_score            0
dtype: int64

Mean of each numeric column:
study_hours_per_day     5.00
attendance_percent     79.35
sleep_hours             6.62
previous_score         66.55
final_score            67

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Adjust paths for Colab environment
# Assumes 'student_performance.csv' and 'student_performance_clean.csv'
# are directly in the current working directory (/content/)
DATA_DIR = Path.cwd()
CLEAN_PATH = DATA_DIR / "student_performance_clean.csv"
RAW_PATH = DATA_DIR / "student_performance.csv"
OUTPUT_DIR = Path.cwd()


def load_dataset() -> pd.DataFrame:
    """Load the cleaned dataset if available, otherwise clean the raw one on the fly."""
    if CLEAN_PATH.exists():
        return pd.read_csv(CLEAN_PATH)

    df = pd.read_csv(RAW_PATH).drop_duplicates()
    numeric_cols = df.select_dtypes(include="number").columns
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())
    return df.reset_index(drop=True)


def prepare_features(df: pd.DataFrame):
    """Select feature columns (X) and target (y). Encodes the categorical column."""
    df = df.copy()
    df["extracurricular_encoded"] = (df["extracurricular"] == "Yes").astype(int)

    feature_cols = [
        "study_hours_per_day",
        "attendance_percent",
        "sleep_hours",
        "previous_score",
        "extracurricular_encoded",
    ]
    X = df[feature_cols]
    y = df["final_score"]
    return X, y, feature_cols


def train_and_evaluate(X, y, feature_cols):
    """Split data, train a Linear Regression model, and evaluate on the test set."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Scaling isn't strictly required for linear regression, but it makes the
    # learned coefficients directly comparable to each other.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("=" * 60)
    print("MODEL EVALUATION (on held-out test set, 20% of data)")
    print("=" * 60)
    print(f"RMSE (Root Mean Squared Error): {rmse:.3f}")
    print(f"MAE  (Mean Absolute Error):     {mae:.3f}")
    print(f"R^2  (Coefficient of Determination): {r2:.3f}")

    print("\nLearned coefficients (standardized features):")
    for name, coef in zip(feature_cols, model.coef_):
        print(f"  {name:28s}: {coef:+.3f}")
    print(f"  {'intercept':28s}: {model.intercept_:.3f}")

    return model, scaler, X_test, y_test, y_pred, rmse


def plot_predictions(y_test, y_pred):
    """Save a predicted-vs-actual scatter plot."""
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.6, edgecolor="k")
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    plt.plot(lims, lims, "r--", label="Perfect prediction")
    plt.xlabel("Actual Final Score")
    plt.ylabel("Predicted Final Score")
    plt.title("Linear Regression: Predicted vs Actual")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "predicted_vs_actual.png", dpi=150)
    plt.close()
    print(f"\nSaved plot to {OUTPUT_DIR / 'predicted_vs_actual.png'}")


def main():
    df = load_dataset()
    X, y, feature_cols = prepare_features(df)
    model, scaler, X_test, y_test, y_pred, rmse = train_and_evaluate(X, y, feature_cols)
    plot_predictions(y_test, y_pred)


if __name__ == "__main__":
    main()

MODEL EVALUATION (on held-out test set, 20% of data)
RMSE (Root Mean Squared Error): 5.807
MAE  (Mean Absolute Error):     4.596
R^2  (Coefficient of Determination): 0.651

Learned coefficients (standardized features):
  study_hours_per_day         : +5.568
  attendance_percent          : +3.822
  sleep_hours                 : +1.818
  previous_score              : +3.676
  extracurricular_encoded     : +0.257
  intercept                   : 67.468

Saved plot to /content/predicted_vs_actual.png


In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Adjust paths for Colab environment
# Assumes 'sentiment_reviews.csv' is directly in the current working directory (/content/)
DATA_PATH = Path.cwd() / "sentiment_reviews.csv"
OUTPUT_DIR = Path.cwd()


def load_data(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)


def build_and_train(df: pd.DataFrame):
    """Split data, vectorize text with TF-IDF, and train a Logistic Regression classifier."""
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        df["text"], df["sentiment"], test_size=0.2, random_state=42, stratify=df["sentiment"]
    )

    vectorizer = TfidfVectorizer(stop_words="english", max_features=2000, ngram_range=(1, 2))
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    return model, vectorizer, X_test_text, y_test, y_pred


def evaluate(y_test, y_pred):
    print("=" * 60)
    print("MODEL EVALUATION (on held-out test set, 20% of data)")
    print("=" * 60)
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
    print(f"Precision: {precision_score(y_test, y_pred, pos_label='positive'):.3f}")
    print(f"Recall:    {recall_score(y_test, y_pred, pos_label='positive'):.3f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred, pos_label='positive'):.3f}")
    print("\nFull classification report:")
    print(classification_report(y_test, y_pred))


def plot_confusion_matrix(y_test, y_pred):
    labels = ["negative", "positive"]
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix - Sentiment Classifier")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
    plt.close()
    print(f"\nSaved confusion matrix to {OUTPUT_DIR / 'confusion_matrix.png'}")


def predict_custom_examples(model, vectorizer):
    """Try the model on a few brand-new, unseen sentences."""
    examples = [
        "This gadget is absolutely wonderful and works better than expected.",
        "I am so disappointed, it broke on the very first day.",
        "Decent product but the delivery was late.",
        "Best purchase I have made this year, love it!",
    ]
    X_new = vectorizer.transform(examples)
    predictions = model.predict(X_new)
    probabilities = model.predict_proba(X_new)

    print("\n" + "=" * 60)
    print("PREDICTIONS ON NEW, UNSEEN EXAMPLES")
    print("=" * 60)
    for text, pred, proba in zip(examples, predictions, probabilities):
        confidence = max(proba)
        print(f"[{pred.upper():8s}] (confidence: {confidence:.2f}) -> {text}")


def main():
    df = load_data(DATA_PATH)
    print(f"Loaded {len(df)} labeled reviews.")
    print(df["sentiment"].value_counts())

    model, vectorizer, X_test_text, y_test, y_pred = build_and_train(df)
    evaluate(y_test, y_pred)
    plot_confusion_matrix(y_test, y_pred)
    predict_custom_examples(model, vectorizer)


if __name__ == "__main__":
    main()

Loaded 392 labeled reviews.
sentiment
positive    198
negative    194
Name: count, dtype: int64
MODEL EVALUATION (on held-out test set, 20% of data)
Accuracy:  1.000
Precision: 1.000
Recall:    1.000
F1 Score:  1.000

Full classification report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        39
    positive       1.00      1.00      1.00        40

    accuracy                           1.00        79
   macro avg       1.00      1.00      1.00        79
weighted avg       1.00      1.00      1.00        79


Saved confusion matrix to /content/confusion_matrix.png

PREDICTIONS ON NEW, UNSEEN EXAMPLES
[POSITIVE] (confidence: 0.67) -> This gadget is absolutely wonderful and works better than expected.
[NEGATIVE] (confidence: 0.76) -> I am so disappointed, it broke on the very first day.
[NEGATIVE] (confidence: 0.61) -> Decent product but the delivery was late.
[POSITIVE] (confidence: 0.63) -> Best purchase I have made this year, 